# ✈️ Atrasos e Cancelamentos de Voos | Engenharia de Features

**Entrada:** `data/raw/flights.csv` · `data/raw/airlines.csv` · `data/raw/airports.csv`  
**Saída:** `data/processed/flights_features.parquet`

## Objetivo

Transformar o dataset bruto de voos em um conjunto de features prontas para os
modelos de **classificação** (atraso ≥ 15 min?) e **clustering** (perfil de causa
de atraso), cobrindo:

- Tratamento de nulos e remoção de escopo (cancelados / desviados).
- Criação de features temporais, de rota e históricas.
- Prevenção de **data leakage** nas features históricas (split antes dos agregados).
- Encoding de variáveis categóricas (fit exclusivo no treino).
- Exportação em Parquet particionado.

## Escopo

- Apenas voos **concluídos** (`CANCELLED = 0`, `DIVERTED = 0`) com `ARRIVAL_DELAY` preenchido.
- Dataset: voos domésticos dos EUA — ano 2015 (Bureau of Transportation Statistics).
- Split temporal: **treino** = meses 1-10 · **teste** = meses 11-12 (consistente com
  o notebook de classificação).

## Setup

In [1]:
import os
import sys
import pandas as pd
import numpy as np
import matplotlib.pyplot as plt
import seaborn as sns

from pyspark.sql import SparkSession, Window
from pyspark.sql import functions as F
from pyspark.sql.types import (
    StructType, StructField,
    StringType, IntegerType, FloatType, DoubleType
)
from pyspark.ml.feature import StringIndexer, VectorAssembler
from pyspark.ml import Pipeline

import warnings

warnings.filterwarnings('ignore')
os.makedirs('../data/processed', exist_ok=True)
sns.set_theme(style='whitegrid', palette='muted', font_scale=1.1)

In [3]:
os.environ['JAVA_HOME'] = "/usr/lib/jvm/java-17-openjdk-amd64"
spark = (
    SparkSession.builder
        .appName("FeatureEngineering")
        .master("local[*]")
        .config("spark.driver.memory", "4g")
        .config("spark.sql.shuffle.partitions", "8")
        .config("spark.sql.repl.eagerEval.enabled", True)
        .getOrCreate()
)
spark.sparkContext.setLogLevel("WARN")
print("="*80)
print(f"Spark Session...\n\tSpark Version: {spark.version}\n\tPython Version: {sys.version}")
print("="*80)

Spark Session...
	Spark Version: 4.1.1
	Python Version: 3.13.5 (main, Jun 11 2025, 15:36:57) [GCC 11.4.0]


In [4]:
PATH_FLIGHTS  = '../data/raw/flights.csv'
PATH_AIRLINES = '../data/raw/airlines.csv'
PATH_AIRPORTS = '../data/raw/airports.csv'
PATH_OUTPUT   = '../data/processed/flights_features.parquet'

os.makedirs('../data/processed', exist_ok=True)

## Data Loading

In [5]:
# Avoids incorrect type inference and improves read performance.
SCHEMA_FLIGHTS = StructType([
    StructField('YEAR',                 IntegerType(), True),
    StructField('MONTH',                IntegerType(), True),
    StructField('DAY',                  IntegerType(), True),
    StructField('DAY_OF_WEEK',          IntegerType(), True),
    StructField('AIRLINE',              StringType(),  True),
    StructField('FLIGHT_NUMBER',        IntegerType(), True),
    StructField('TAIL_NUMBER',          StringType(),  True),
    StructField('ORIGIN_AIRPORT',       StringType(),  True),
    StructField('DESTINATION_AIRPORT',  StringType(),  True),
    StructField('SCHEDULED_DEPARTURE',  IntegerType(), True),
    StructField('DEPARTURE_TIME',       FloatType(),   True),
    StructField('DEPARTURE_DELAY',      FloatType(),   True),
    StructField('TAXI_OUT',             FloatType(),   True),
    StructField('WHEELS_OFF',           FloatType(),   True),
    StructField('SCHEDULED_TIME',       FloatType(),   True),
    StructField('ELAPSED_TIME',         FloatType(),   True),
    StructField('AIR_TIME',             FloatType(),   True),
    StructField('DISTANCE',             FloatType(),   True),
    StructField('WHEELS_ON',            FloatType(),   True),
    StructField('TAXI_IN',              FloatType(),   True),
    StructField('SCHEDULED_ARRIVAL',    IntegerType(), True),
    StructField('ARRIVAL_TIME',         FloatType(),   True),
    StructField('ARRIVAL_DELAY',        FloatType(),   True),
    StructField('DIVERTED',             IntegerType(), True),
    StructField('CANCELLED',            IntegerType(), True),
    StructField('CANCELLATION_REASON',  StringType(),  True),
    StructField('AIR_SYSTEM_DELAY',     FloatType(),   True),
    StructField('SECURITY_DELAY',       FloatType(),   True),
    StructField('AIRLINE_DELAY',        FloatType(),   True),
    StructField('LATE_AIRCRAFT_DELAY',  FloatType(),   True),
    StructField('WEATHER_DELAY',        FloatType(),   True),
])

df = (
    spark.read
    .option('header', 'true')
    .option('nullValue', 'NA')
    .schema(SCHEMA_FLIGHTS)
    .csv(PATH_FLIGHTS)
)

df_airlines = spark.read.option('header', 'true').csv(PATH_AIRLINES)
df_airports = spark.read.option('header', 'true').csv(PATH_AIRPORTS)

print(f'Registros carregados: {df.count():,}')
print(f'Colunas: {len(df.columns)}')

Registros carregados: 5,819,079
Colunas: 31


## Definição de Escopo

> O modelo supervisionado deve prever o atraso na **chegada** dos voos que
> **foram concluídos**. Portanto, voos desviados e cancelados são removidos.
>
> Registros sem `ARRIVAL_DELAY` informado também são descartados.

In [6]:
n_before = df.count()

df = df.filter(
    (F.col('CANCELLED') == 0) &
    (F.col('DIVERTED')  == 0) &
    (F.col('ARRIVAL_DELAY').isNotNull())
)

n_after = df.count()
print(f'Removidos (cancelados/desviados/sem ARRIVAL_DELAY): {n_before - n_after:,}')
print(f'Registros no escopo do modelo: {n_after:,}')

df = df.drop('CANCELLED', 'DIVERTED', 'CANCELLATION_REASON')
print(f'Colunas restantes: {len(df.columns)}')

Removidos (cancelados/desviados/sem ARRIVAL_DELAY): 105,071
Registros no escopo do modelo: 5,714,008
Colunas restantes: 28


## Null Values

### Delay Cause Columns

**Colunas:** `AIR_SYSTEM_DELAY`, `SECURITY_DELAY`, `AIRLINE_DELAY`,
`LATE_AIRCRAFT_DELAY`, `WEATHER_DELAY`

**Hipótese (H₀):**  
O DOT **não** preenche as colunas de causa quando `ARRIVAL_DELAY < 15`.
Portanto, nulos nessas colunas representam ausência de atraso registrado
(equivalente a 0 minutos), e não dados faltantes de verdade.

In [7]:
DELAY_CAUSE_COLS = [
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY'
]

In [8]:
df_on_time = df.filter(F.col('ARRIVAL_DELAY') <  15)
df_delayed = df.filter(F.col('ARRIVAL_DELAY') >= 15)

n_on_time = df_on_time.count()
n_delayed = df_delayed.count()

print(f'Voos pontuais  (ARRIVAL_DELAY <  15): {n_on_time:>9,}')
print(f'Voos atrasados (ARRIVAL_DELAY >= 15): {n_delayed:>9,}')
print()

for col in DELAY_CAUSE_COLS:
    nulls_on_time = df_on_time.filter(F.col(col).isNull()).count()
    nulls_delayed = df_delayed.filter(F.col(col).isNull()).count()
    print(f'  {col:<22} | pontuais: {nulls_on_time/n_on_time*100:>6.2f}% nulos | '
          f'atrasados: {nulls_delayed/n_delayed*100:>6.2f}% nulos')

Voos pontuais  (ARRIVAL_DELAY <  15): 4,650,569
Voos atrasados (ARRIVAL_DELAY >= 15): 1,063,439



  AIR_SYSTEM_DELAY       | pontuais: 100.00% nulos | atrasados:   0.00% nulos


  SECURITY_DELAY         | pontuais: 100.00% nulos | atrasados:   0.00% nulos


  AIRLINE_DELAY          | pontuais: 100.00% nulos | atrasados:   0.00% nulos


  LATE_AIRCRAFT_DELAY    | pontuais: 100.00% nulos | atrasados:   0.00% nulos


  WEATHER_DELAY          | pontuais: 100.00% nulos | atrasados:   0.00% nulos


In [9]:
anomaly_condition = ' OR '.join([f'{c} IS NOT NULL' for c in DELAY_CAUSE_COLS])
n_anomalies = df_on_time.filter(anomaly_condition).count()

print(f'Voos pontuais (ARRIVAL_DELAY < 15 min) com causa preenchida: {n_anomalies:,}')
if n_anomalies == 0:
    print('Hipótese confirmada.')
else:
    print(f'Hipótese rejeitada: {n_anomalies:,} registros anormais detectados.')

Voos pontuais (ARRIVAL_DELAY < 15 min) com causa preenchida: 0
Hipótese confirmada.


In [10]:
# LABEL = 1 → arrival delay ≥ 15 min  |  LABEL = 0 → on-time
df = df.withColumn(
    'LABEL',
    F.when(F.col('ARRIVAL_DELAY') >= 15, 1).otherwise(0).cast(IntegerType())
)

In [11]:
# ARRIVAL_DELAY <  15  →  DOT does not record cause  →  null = 0 min
# ARRIVAL_DELAY >= 15  →  DOT records cause        →  real value
df = df.fillna(0, subset=DELAY_CAUSE_COLS)

In [12]:
remaining_nulls = {c: df.filter(F.col(c).isNull()).count() for c in DELAY_CAUSE_COLS}
print('Nulos residuais após fillna(0):', remaining_nulls)

Nulos residuais após fillna(0): {'AIR_SYSTEM_DELAY': 0, 'SECURITY_DELAY': 0, 'AIRLINE_DELAY': 0, 'LATE_AIRCRAFT_DELAY': 0, 'WEATHER_DELAY': 0}


In [13]:
n_total = df.count()
label_dist = (
    df.groupBy('LABEL').count()
    .withColumn('PCT', F.round(F.col('count') / n_total * 100, 2))
    .orderBy('LABEL')
    .toPandas()
)
label_dist['DESC'] = label_dist['LABEL'].map(
    {0: 'Não atrasado (< 15 min)', 1: 'Atrasado (>= 15 min)'}
)
print('Distribuição final da variável-alvo:')
print(label_dist[['DESC', 'count', 'PCT']].to_string(index=False))
print('\nRatio 0:1 =', round(label_dist.loc[0, 'count'] / label_dist.loc[1, 'count'], 2))

# Cause columns are leakage in the predictive feature set
LEAKAGE_COLS = DELAY_CAUSE_COLS.copy()
print(f'\nColunas marcadas como LEAKAGE: {LEAKAGE_COLS}')

Distribuição final da variável-alvo:
                   DESC   count   PCT
Não atrasado (< 15 min) 4650569 81.39
   Atrasado (>= 15 min) 1063439 18.61

Ratio 0:1 = 4.37

Colunas marcadas como LEAKAGE: ['AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY']


### Colunas Operacionais

Como visto na EDA, apenas `ELAPSED_TIME` e `SCHEDULED_TIME` apresentam
percentual de nulos significativo. Registros com nulo nessas colunas são
removidos — representam voos com dados operacionais incompletos.

In [14]:
n_before = df.count()

df = df.filter(
    F.col('ELAPSED_TIME').isNotNull() &
    F.col('SCHEDULED_TIME').isNotNull()
)

n_after   = df.count()
n_removed = n_before - n_after

print(f'Registros antes : {n_before:,}')
print(f'Registros depois: {n_after:,}')
print(f'Removidos       : {n_removed:,} ({n_removed/n_before*100:.2f}%)')

OPERATIONAL_COLS = [
    'DEPARTURE_DELAY', 'DEPARTURE_TIME', 'TAXI_OUT', 'TAXI_IN',
    'ELAPSED_TIME', 'AIR_TIME', 'WHEELS_OFF', 'WHEELS_ON',
    'ARRIVAL_TIME', 'SCHEDULED_TIME'
]

print('\nNulos residuais nas colunas operacionais:')
for c in OPERATIONAL_COLS:
    n = df.filter(F.col(c).isNull()).count()
    status = 'OK' if n == 0 else f'ATENCAO  {n:,}'
    print(f'  {c:<22}: {status}')

Registros antes : 5,714,008
Registros depois: 5,714,008
Removidos       : 0 (0.00%)

Nulos residuais nas colunas operacionais:


  DEPARTURE_DELAY       : OK


  DEPARTURE_TIME        : OK
  TAXI_OUT              : OK


  TAXI_IN               : OK


  ELAPSED_TIME          : OK


  AIR_TIME              : OK
  WHEELS_OFF            : OK


  WHEELS_ON             : OK


  ARRIVAL_TIME          : OK
  SCHEDULED_TIME        : OK


### Remoção de Colunas Irrelevantes

Colunas sem sinal preditivo generalizável são descartadas antes do
feature engineering para reduzir o volume de dados em memória.

In [15]:
DROP_COLS = {
    'YEAR'         : 'Constante (todos os registros são de 2015) — variância zero',
    'TAIL_NUMBER'  : 'Alta cardinalidade sem padrão generalizável por aeronave',
    'FLIGHT_NUMBER': 'Identificador operacional — não carrega sinal preditivo estável',
}

for col, reason in DROP_COLS.items():
    print(f'  DROP | {col:<15} → {reason}')

df = df.drop(*DROP_COLS.keys())
print(f'\nColunas restantes: {len(df.columns)}')
print(df.columns)

  DROP | YEAR            → Constante (todos os registros são de 2015) — variância zero
  DROP | TAIL_NUMBER     → Alta cardinalidade sem padrão generalizável por aeronave
  DROP | FLIGHT_NUMBER   → Identificador operacional — não carrega sinal preditivo estável

Colunas restantes: 26
['MONTH', 'DAY', 'DAY_OF_WEEK', 'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'SCHEDULED_DEPARTURE', 'DEPARTURE_TIME', 'DEPARTURE_DELAY', 'TAXI_OUT', 'WHEELS_OFF', 'SCHEDULED_TIME', 'ELAPSED_TIME', 'AIR_TIME', 'DISTANCE', 'WHEELS_ON', 'TAXI_IN', 'SCHEDULED_ARRIVAL', 'ARRIVAL_TIME', 'ARRIVAL_DELAY', 'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY', 'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY', 'LABEL']


In [16]:
n = df.count()
label_dist = (
    df.groupBy('LABEL').count()
    .withColumn('PCT', F.round(F.col('count') / n * 100, 2))
    .orderBy('LABEL')
    .toPandas()
)
label_dist['DESC'] = label_dist['LABEL'].map(
    {0: 'Pontual (< 15 min)', 1: 'Atrasado (>= 15 min)'}
)
print(f'Total de registros: {n:,}')
print(label_dist[['DESC', 'count', 'PCT']].to_string(index=False))

Total de registros: 5,714,008
                DESC   count   PCT
  Pontual (< 15 min) 4650569 81.39
Atrasado (>= 15 min) 1063439 18.61


## Engenharia de Features

### Features Temporais

`SCHEDULED_DEPARTURE` e `SCHEDULED_ARRIVAL` estão no formato HHMM
(ex: 830 = 08h30). Dividir por 100 e truncar extrai a hora inteira.

In [17]:
# Exemplos: 830 → 08h30  |  1245 → 12h45  |  2359 → 23h59
df = df.withColumn('HORA_PARTIDA',
        (F.col('SCHEDULED_DEPARTURE') / 100).cast(IntegerType()))

df = df.withColumn('HORA_CHEGADA_PROG',
        (F.col('SCHEDULED_ARRIVAL') / 100).cast(IntegerType()))


In [18]:
# Captures the cascade delay pattern in a non-linear way
df = df.withColumn('TURNO',
    F.when(F.col('HORA_PARTIDA').between(5,  11), F.lit('manha'))
    .when(F.col('HORA_PARTIDA').between(12, 17), F.lit('tarde'))
    .when(F.col('HORA_PARTIDA').between(18, 21), F.lit('noite'))
    .otherwise(F.lit('madrugada'))
)

In [19]:
# Binary flags
df = df.withColumn('IS_WEEKEND',
    F.when(F.col('DAY_OF_WEEK').isin([6, 7]), 1).otherwise(0).cast(IntegerType()))

df = df.withColumn('IS_PEAK_MONTH',
    F.when(F.col('MONTH').isin([6, 7, 12]), 1).otherwise(0).cast(IntegerType()))

In [20]:
print('Features temporais criadas:')
df.select(
    'SCHEDULED_DEPARTURE', 'HORA_PARTIDA',
    'SCHEDULED_ARRIVAL', 'HORA_CHEGADA_PROG',
    'TURNO', 'IS_WEEKEND', 'IS_PEAK_MONTH'
).show(5, truncate=False)

Features temporais criadas:
+-------------------+------------+-----------------+-----------------+---------+----------+-------------+
|SCHEDULED_DEPARTURE|HORA_PARTIDA|SCHEDULED_ARRIVAL|HORA_CHEGADA_PROG|TURNO    |IS_WEEKEND|IS_PEAK_MONTH|
+-------------------+------------+-----------------+-----------------+---------+----------+-------------+
|5                  |0           |430              |4                |madrugada|0         |0            |
|10                 |0           |750              |7                |madrugada|0         |0            |
|20                 |0           |806              |8                |madrugada|0         |0            |
|20                 |0           |805              |8                |madrugada|0         |0            |
|25                 |0           |320              |3                |madrugada|0         |0            |
+-------------------+------------+-----------------+-----------------+---------+----------+-------------+
only showing top 5

### Features de Rota

Origem-destino como identificador único de rota, frequência da rota
(característica estrutural — sem risco de leakage) e log da distância
(reduz o peso de rotas muito longas na distribuição assimétrica).

In [21]:
df = df.withColumn('ROTA',
    F.concat(F.col('ORIGIN_AIRPORT'), F.lit('_'), F.col('DESTINATION_AIRPORT')))

df = df.withColumn('LOG_DISTANCE', F.log1p(F.col('DISTANCE')))

print('Route features created:')
df.select('ROTA', 'DISTANCE', 'LOG_DISTANCE').show(5)

Route features created:
+-------+--------+-----------------+
|   ROTA|DISTANCE|     LOG_DISTANCE|
+-------+--------+-----------------+
|ANC_SEA|  1448.0|7.278628942320682|
|LAX_PBI|  2330.0|7.754052639035757|
|SFO_CLT|  2296.0|7.739359202689098|
|LAX_MIA|  2342.0|7.759187438507795|
|SEA_ANC|  1448.0|7.278628942320682|
+-------+--------+-----------------+
only showing top 5 rows


### Features Históricas

Atraso médio histórico por grupo (companhia, aeroporto de origem,
aeroporto de destino, rota).

> **⚠️ Atenção — Data Leakage**  
> As features são derivadas de `ARRIVAL_DELAY`, a própria variável alvo.  
> Se calculadas sobre o dataset completo, os dados de **teste (meses 11-12)**
> contaminam as médias usadas no treino — o modelo aprenderia padrões que
> não existiriam no momento real da predição.  
>  
> **Correção:**  
> 1. Separar `df_train` (meses 1-10) e `df_test` (meses 11-12) **antes** do cálculo.  
> 2. Calcular as médias históricas **somente sobre `df_train`**.  
> 3. Aplicar as médias no teste via join (grupos ausentes → imputação pela média
>    global do treino).  
> 4. Reunificar em `df` para que o restante do notebook continue sem alterações.

In [22]:
df_train = df.filter(F.col('MONTH') <= 10)
df_test  = df.filter(F.col('MONTH') >  10)

print(f'Train (months  1-10): {df_train.count():>10,} records')
print(f'Test  (months 11-12): {df_test.count():>10,} records')

# FREQ_ROTA: computed on training set only to prevent leakage
route_freq = (
    df_train.groupBy('ROTA')
    .count()
    .withColumnRenamed('count', 'FREQ_ROTA')
)
global_freq = df_train.count() / df_train.select('ROTA').distinct().count()

df_train = df_train.join(route_freq, on='ROTA', how='left')
df_test  = df_test.join(route_freq, on='ROTA', how='left')
df_test  = df_test.withColumn('FREQ_ROTA', F.coalesce(F.col('FREQ_ROTA'), F.lit(global_freq)))


def add_hist_delay(df_ref, df_target, group_cols, new_col):
    if isinstance(group_cols, str):
        group_cols = [group_cols]
    agg = (
        df_ref.groupBy(group_cols)
        .agg(F.round(F.mean('ARRIVAL_DELAY'), 3).alias(new_col))
    )
    return df_target.join(agg, on=group_cols, how='left')


HIST_GROUPS = [
    ('AIRLINE',                  'HIST_DELAY_AIRLINE'),
    ('ORIGIN_AIRPORT',           'HIST_DELAY_ORIGIN'),
    ('DESTINATION_AIRPORT',      'HIST_DELAY_DEST'),
    ('ROTA',                     'HIST_DELAY_ROTA'),
    (['AIRLINE', 'DAY_OF_WEEK'], 'HIST_DELAY_AIRLINE_DOW'),
    (['AIRLINE', 'TURNO'],       'HIST_DELAY_AIRLINE_TURNO'),
]

for group_cols, col_name in HIST_GROUPS:
    df_train = add_hist_delay(df_train, df_train, group_cols, col_name)
    df_test  = add_hist_delay(df_train, df_test,  group_cols, col_name)


HIST_COLS = [col for _, col in HIST_GROUPS]
global_means = (
    df_train
    .agg(*[F.mean(c).alias(c) for c in HIST_COLS])
    .first()
    .asDict()
)

for c in HIST_COLS:
    df_test = df_test.withColumn(
        c, F.coalesce(F.col(c), F.lit(global_means[c]))
    )


df = df_train.union(df_test)

print('\nHistorical features created (no leakage):')
df.select(*HIST_COLS).show(5)

Train (months  1-10):  4,781,924 records
Test  (months 11-12):    932,084 records



Historical features created (no leakage):


+------------------+-----------------+---------------+---------------+----------------------+------------------------+
|HIST_DELAY_AIRLINE|HIST_DELAY_ORIGIN|HIST_DELAY_DEST|HIST_DELAY_ROTA|HIST_DELAY_AIRLINE_DOW|HIST_DELAY_AIRLINE_TURNO|
+------------------+-----------------+---------------+---------------+----------------------+------------------------+
|             0.296|            5.317|           3.63|         -2.114|                 1.849|                   1.791|
|             3.706|            5.317|          2.358|         -2.597|                 4.527|                   4.447|
|            -0.799|           -0.616|          1.983|         -2.924|                -0.064|                  -4.748|
|             3.705|            6.197|          6.881|         10.052|                 5.468|                   3.053|
|             3.705|            6.197|           6.38|          6.313|                 5.468|                   3.053|
+------------------+-----------------+----------

## Feature Validation

Verifica que todas as features criadas estão livres de nulos antes
de prosseguir com o encoding e a seleção final.

In [23]:
NEW_FEATURES = [
    'HORA_PARTIDA', 'HORA_CHEGADA_PROG', 'TURNO', 'IS_WEEKEND', 'IS_PEAK_MONTH',
    'ROTA', 'FREQ_ROTA', 'LOG_DISTANCE',
    'HIST_DELAY_AIRLINE', 'HIST_DELAY_ORIGIN', 'HIST_DELAY_DEST',
    'HIST_DELAY_ROTA', 'HIST_DELAY_AIRLINE_DOW', 'HIST_DELAY_AIRLINE_TURNO'
]

print(f'Features criadas: {len(NEW_FEATURES)}')
print(f'Colunas totais:   {len(df.columns)}\n')

print('Nulos nas novas features:')
for c in NEW_FEATURES:
    n = df.filter(F.col(c).isNull()).count()
    status = 'OK  0' if n == 0 else f'ATENCAO  {n:,}'
    print(f'  {c:<28}: {status}')

Features criadas: 14
Colunas totais:   40

Nulos nas novas features:


  HORA_PARTIDA                : OK  0


  HORA_CHEGADA_PROG           : OK  0
  TURNO                       : OK  0
  IS_WEEKEND                  : OK  0
  IS_PEAK_MONTH               : OK  0


  ROTA                        : OK  0


  FREQ_ROTA                   : OK  0


...

  HIST_DELAY_ORIGIN           : OK  0


  HIST_DELAY_DEST             : OK  0


  HIST_DELAY_ROTA             : OK  0


  HIST_DELAY_AIRLINE_DOW      : OK  0


  HIST_DELAY_AIRLINE_TURNO    : OK  0


## Encoding

Variáveis categóricas são convertidas para índices numéricos com
`StringIndexer`. O mapeamento string→índice é **aprendido exclusivamente
no treino** (`MONTH <= 10`) para evitar que categorias do teste influenciem
a codificação. `handleInvalid='keep'` garante que categorias inéditas no
teste recebam um índice extra em vez de causar erro.

In [24]:
CATEGORICAL_COLS = ['AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'TURNO']

indexers = [
    StringIndexer(
        inputCol=c,
        outputCol=f'{c}_IDX',
        handleInvalid='keep'
    )
    for c in CATEGORICAL_COLS
]

# Fit on training set only
encoder = Pipeline(stages=indexers).fit(df.filter(F.col('MONTH') <= 10))
df = encoder.transform(df)

print('Encoding aplicado:')
df.select([f'{c}_IDX' for c in CATEGORICAL_COLS]).show(5)

Encoding aplicado:
+-----------+------------------+-----------------------+---------+
|AIRLINE_IDX|ORIGIN_AIRPORT_IDX|DESTINATION_AIRPORT_IDX|TURNO_IDX|
+-----------+------------------+-----------------------+---------+
|        9.0|              66.0|                   10.0|      3.0|
|        2.0|               4.0|                   54.0|      3.0|
|        8.0|               7.0|                   14.0|      3.0|
|        2.0|               4.0|                   24.0|      3.0|
|        9.0|              10.0|                   66.0|      3.0|
+-----------+------------------+-----------------------+---------+
only showing top 5 rows


## Final Selection and Export

In [25]:
CLASSIFIER_FEATURES = [
    # Temporal
    'MONTH', 'DAY_OF_WEEK', 'HORA_PARTIDA', 'HORA_CHEGADA_PROG',
    'IS_WEEKEND', 'IS_PEAK_MONTH',
    # Route and distance
    'LOG_DISTANCE', 'FREQ_ROTA',
    # Indexed categorical
    'AIRLINE_IDX', 'ORIGIN_AIRPORT_IDX', 'DESTINATION_AIRPORT_IDX', 'TURNO_IDX',
    # Historical
    'HIST_DELAY_AIRLINE', 'HIST_DELAY_ORIGIN', 'HIST_DELAY_DEST',
    'HIST_DELAY_ROTA', 'HIST_DELAY_AIRLINE_DOW', 'HIST_DELAY_AIRLINE_TURNO',
]

CLUSTERING_FEATURES = [
    'AIRLINE', 'ORIGIN_AIRPORT', 'DESTINATION_AIRPORT', 'ROTA',
    'ARRIVAL_DELAY', 'DEPARTURE_DELAY',
    'AIR_SYSTEM_DELAY', 'SECURITY_DELAY', 'AIRLINE_DELAY',
    'LATE_AIRCRAFT_DELAY', 'WEATHER_DELAY',
]

FINAL_COLS = CLASSIFIER_FEATURES + ['LABEL'] + CLUSTERING_FEATURES

df_output = df.select(FINAL_COLS)

print(f'Features do modelo : {len(CLASSIFIER_FEATURES)}')
print(f'Colunas no Parquet : {len(FINAL_COLS)}')

print(f'\nColunas descartadas nesta etapa:')
dropped_cols = [c for c in df.columns if c not in FINAL_COLS]
for c in dropped_cols:
    print(f'  {c}')

Features do modelo : 18
Colunas no Parquet : 30

Colunas descartadas nesta etapa:
  TURNO
  DAY
  SCHEDULED_DEPARTURE
  DEPARTURE_TIME
  TAXI_OUT
  WHEELS_OFF
  SCHEDULED_TIME
  ELAPSED_TIME
  AIR_TIME
  DISTANCE
  WHEELS_ON
  TAXI_IN
  SCHEDULED_ARRIVAL
  ARRIVAL_TIME


In [26]:
print('Verificando nulos nas features do modelo...')
null_counts = {
    c: df_output.filter(F.col(c).isNull()).count()
    for c in CLASSIFIER_FEATURES
}
issues = {k: v for k, v in null_counts.items() if v > 0}

if not issues:
    print('Zero nulos nas features do modelo\n')
else:
    print(f'Nulos encontrados: {issues}')

Verificando nulos nas features do modelo...


Zero nulos nas features do modelo



In [27]:
(
    df_output
    .repartition(4)
    .write
    .mode('overwrite')
    .parquet('../data/processed/flights_features.parquet')
)

26/04/05 11:35:54 WARN SparkStringUtils: Truncated the string representation of a plan since it was too large. This behavior can be adjusted by setting 'spark.sql.debug.maxToStringFields'.


In [28]:
df_validation = spark.read.parquet('../data/processed/flights_features.parquet')
print(f'Registros salvos : {df_validation.count():,}')
print(f'Colunas          : {len(df_validation.columns)}')
print(f'\nSchema final:')
df_validation.printSchema()

Registros salvos : 5,714,008
Colunas          : 30

Schema final:
root
 |-- MONTH: integer (nullable = true)
 |-- DAY_OF_WEEK: integer (nullable = true)
 |-- HORA_PARTIDA: integer (nullable = true)
 |-- HORA_CHEGADA_PROG: integer (nullable = true)
 |-- IS_WEEKEND: integer (nullable = true)
 |-- IS_PEAK_MONTH: integer (nullable = true)
 |-- LOG_DISTANCE: double (nullable = true)
 |-- FREQ_ROTA: double (nullable = true)
 |-- AIRLINE_IDX: double (nullable = true)
 |-- ORIGIN_AIRPORT_IDX: double (nullable = true)
 |-- DESTINATION_AIRPORT_IDX: double (nullable = true)
 |-- TURNO_IDX: double (nullable = true)
 |-- HIST_DELAY_AIRLINE: double (nullable = true)
 |-- HIST_DELAY_ORIGIN: double (nullable = true)
 |-- HIST_DELAY_DEST: double (nullable = true)
 |-- HIST_DELAY_ROTA: double (nullable = true)
 |-- HIST_DELAY_AIRLINE_DOW: double (nullable = true)
 |-- HIST_DELAY_AIRLINE_TURNO: double (nullable = true)
 |-- LABEL: integer (nullable = true)
 |-- AIRLINE: string (nullable = true)
 |-- ORIG